# Notebook 03 — Fine-tune DistilBERT Intent Classifier

**WHERE IT RUNS**: Google Colab or Kaggle with T4 GPU  
**EXPECTED RUNTIME**: ~25 min for 10k labeled examples  
**OUTPUT**: `models/classifier/` — download and place in your local repo  

## Steps
1. Install dependencies
2. Run fine-tuning using Kaggle Input data
3. Download the model from Kaggle Output


In [ ]:
# ── Sanity Check ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'GPU not available! Switch to GPU runtime: Runtime > Change runtime type > T4 GPU'
print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q transformers datasets evaluate accelerate scikit-learn pyyaml tqdm

In [ ]:
# Kaggle Dataset Paths
import os

# Dynamically find the dataset directory in Kaggle
DATASET_DIR = '/kaggle/input'
for root, dirs, files in os.walk('/kaggle/input'):
    if 'intent_taxonomy.yaml' in files:
        DATASET_DIR = root
        break

TAXONOMY_PATH = f'{DATASET_DIR}/intent_taxonomy.yaml'
DATA_PATH = f'{DATASET_DIR}/labeled_training.csv'
print(f"Using dataset directory: {DATASET_DIR}")


In [ ]:
import pandas as pd
import yaml
import numpy as np
from pathlib import Path
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
import evaluate

# Config
BASE_MODEL = 'distilbert-base-uncased'
MAX_LENGTH = 128
NUM_EPOCHS = 4
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EVAL_SPLIT = 0.1
RANDOM_SEED = 42
MODEL_OUT = '/kaggle/working/models/classifier'

# Load taxonomy
with open(TAXONOMY_PATH) as f:
    taxonomy_data = yaml.safe_load(f)
intent_names = [i['name'] for i in taxonomy_data['intents']]
label2id = {n: i for i, n in enumerate(intent_names)}
id2label = {i: n for n, i in label2id.items()}
print(f'Taxonomy: {len(intent_names)} intents → {intent_names}')

In [ ]:
# Load labeled data
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['text', 'intent'])
df = df[df['intent'].isin(label2id)]
df['label'] = df['intent'].map(label2id)

print(f'Loaded {len(df):,} examples')
print(f'Label distribution:\n{df["intent"].value_counts()}')

In [ ]:
# Train/eval split
train_df, eval_df = train_test_split(df, test_size=EVAL_SPLIT, random_state=RANDOM_SEED, stratify=df['label'])
train_dataset = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df[['text', 'label']].reset_index(drop=True))

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

print(f'Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}')

In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(intent_names),
    id2label=id2label, label2id=label2id
)

# Metrics
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average='macro')
    return {'accuracy': acc['accuracy'], 'f1_macro': f1['f1']}

# Training
Path(MODEL_OUT).mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=MODEL_OUT,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=50,
    report_to='none',
    seed=RANDOM_SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Save & evaluate
trainer.save_model(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
metrics = trainer.evaluate()
print(f'\nFinal metrics: {metrics}')
print(f'Model saved to {MODEL_OUT}')

In [ ]:
# Download the model as a zip
import shutil
from IPython.display import FileLink

shutil.make_archive('/kaggle/working/classifier_model', 'zip', '/kaggle/working/models/classifier')
print('Model zipped! Download it from the Kaggle Data/Output pane on the right, or click the link below:')
display(FileLink(r'classifier_model.zip'))